# Model 3 — Conditional VAE Date Generator
Run on a **T4 GPU** Colab: Runtime → Change runtime type → T4 GPU

In [ ]:
# ── 1. Clone & install ────────────────────────────────────────────────────────
REPO = "https://github.com/SalmaSherif7070/Conditional-Date-Generation-Using-Deep-Generative-Models"
!git clone {REPO} repo
%cd repo
!pip install -q -r requirements.txt

In [ ]:
# ── 2. GPU check ─────────────────────────────────────────────────────────────
import torch
print('PyTorch:', torch.__version__)
print('GPU    :', torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else 'NOT available — set Runtime → T4 GPU')

In [ ]:
# ── 3. Create __init__.py files ───────────────────────────────────────────────
import os
for pkg in ['src', 'src/model_1', 'src/model_2', 'src/model_3', 'src/model_4']:
    os.makedirs(pkg, exist_ok=True)
    p = os.path.join(pkg, '__init__.py')
    if not os.path.exists(p):
        open(p, 'w').close()
print('✓ __init__.py ready')

In [ ]:
# ── 4. Patch config.py — 100 epochs, flat output paths ───────────────────────
config_src = '''
from dataclasses import dataclass

@dataclass
class ModelConfig:
    cond_dim: int = 128
    max_decade: int = 300

@dataclass
class TrainConfig:
    n_epochs: int = 100
    batch_size: int = 256
    lr: float = 2e-3
    val_split: float = 0.2
    seed: int = 42

@dataclass
class Model2Config:
    cond_dim: int = 128
    max_decade: int = 300
    z_dim: int = 64

@dataclass
class Train2Config:
    n_epochs: int = 100
    batch_size: int = 256
    lr_g: float = 1e-4
    lr_d: float = 4e-4
    n_critic: int = 2
    lambda_gp: float = 10.0
    tau_start: float = 2.0
    tau_end: float = 0.5
    val_split: float = 0.2
    seed: int = 42

@dataclass
class Model3Config:
    cond_dim: int = 128
    max_decade: int = 300
    z_dim: int = 64

@dataclass
class Train3Config:
    n_epochs: int = 100
    batch_size: int = 256
    lr: float = 1e-3
    beta_max: float = 0.5
    beta_warmup_frac: float = 0.5
    val_split: float = 0.2
    seed: int = 42

@dataclass
class Model4Config:
    cond_dim: int = 128
    max_decade: int = 300
    hidden_dim: int = 512
    n_layers: int = 4

@dataclass
class Train4Config:
    n_epochs: int = 100
    batch_size: int = 256
    lr: float = 1e-4
    n_mcmc_steps: int = 60
    mcmc_step_size: float = 0.1
    mcmc_noise: float = 0.005
    replay_buffer_size: int = 10_000
    replay_prob: float = 0.95
    l2_reg: float = 1.0
    grad_clip: float = 1.0
    val_split: float = 0.2
    seed: int = 42

@dataclass
class PathConfig:
    data_path: str = "data/raw/data.txt"
    example_input_path: str = "data/raw/example_input.txt"
    output_dir: str = "output"
    weights_path: str = "output/weights.pt"
    figures_dir: str = "output/figures"

@dataclass
class Path2Config:
    data_path: str = "data/raw/data.txt"
    example_input_path: str = "data/raw/example_input.txt"
    output_dir: str = "output"
    generator_path: str = "output/generator.pt"
    discriminator_path: str = "output/discriminator.pt"
    encoder_path: str = "output/encoder.pt"
    figures_dir: str = "output/figures"

@dataclass
class Path3Config:
    data_path: str = "data/raw/data.txt"
    example_input_path: str = "data/raw/example_input.txt"
    output_dir: str = "output"
    weights_path: str = "output/weights.pt"
    figures_dir: str = "output/figures"

@dataclass
class Path4Config:
    data_path: str = "data/raw/data.txt"
    example_input_path: str = "data/raw/example_input.txt"
    output_dir: str = "output"
    weights_path: str = "output/weights.pt"
    figures_dir: str = "output/figures"
'''
with open('src/config.py', 'w') as f:
    f.write(config_src.strip())
print('✓ src/config.py patched')

In [ ]:
# ── 5. Create output directories ─────────────────────────────────────────────
import os
os.makedirs('output/figures', exist_ok=True)
print('✓ Directories ready')

In [ ]:
# ── 6. Train ──────────────────────────────────────────────────────────────────
!python main.py train3

In [ ]:
# ── 7. Evaluate ───────────────────────────────────────────────────────────────
!python main.py evaluate3

In [ ]:
# ── 8. Predict ────────────────────────────────────────────────────────────────
!python main.py predict3 \
    -i data/raw/example_input.txt \
    -o output/predictions.txt

print('\nFirst 10 predictions:')
with open('output/predictions.txt') as f:
    for i, line in enumerate(f):
        if i >= 10: break
        print(line, end='')

In [ ]:
# ── 9. Display figures ────────────────────────────────────────────────────────
from IPython.display import Image, display
import glob
figs = sorted(glob.glob('output/figures/*.png'))
print(f'Found {len(figs)} figures:')
for path in figs:
    print('\n', path)
    display(Image(path))

In [ ]:
# ── 10. Zip & download ────────────────────────────────────────────────────────
import zipfile, os
ZIP = 'output/model3_outputs.zip'
with zipfile.ZipFile(ZIP, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fn in sorted(os.listdir('output/figures')):
        zf.write(f'output/figures/{fn}', f'figures/{fn}')
    if os.path.exists('output/predictions.txt'):
        zf.write('output/predictions.txt', 'predictions.txt')
    if os.path.exists('output/weights.pt'):
        zf.write('output/weights.pt', 'weights.pt')
print(f'✓ ZIP ({os.path.getsize(ZIP)/1e6:.1f} MB) → {ZIP}')
from google.colab import files
files.download(ZIP)